# MedlinePlus RAG Prototype

This notebook tests whether MedlinePlus passages can provide useful plain-language context for FaithfulMed's Simplifier.

The clinical source remains the authority for patient-specific facts. Retrieved passages may explain medical terms, but must not add diagnoses, treatment instructions, or other patient-specific claims.

In [3]:
%pip install google-genai chromadb pandas python-dotenv
import os
from pathlib import Path

import chromadb
import pandas as pd
from dotenv import load_dotenv
from google import genai

Note: you may need to restart the kernel to use updated packages.


In [2]:
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "medlineplus_seed.csv"

## Load the MedlinePlus passages

Create `data/medlineplus_seed.csv` with one passage per row and these columns:

- `title`: page or section title
- `text`: a short passage copied or summarized according to the source's terms
- `url`: the MedlinePlus source URL

Keep the source URL with every passage so retrieval results can be checked.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Create {DATA_PATH} with columns title, text, and url before continuing."
    )

## Split passages into chunks

In [ ]:
documents_df = pd.read_csv(DATA_PATH).dropna(subset=["title", "text", "url"])
documents_df["text"] = documents_df["text"].astype(str).str.strip()
documents_df = documents_df[documents_df["text"] != ""].reset_index(drop=True)

display(documents_df.head())
print(f"Loaded {len(documents_df)} passages.")

In [ ]:
def chunk_text(text: str, chunk_words: int = 180, overlap_words: int = 30) -> list[str]:
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap_words
    return chunks

In [ ]:
chunks = []
for row_index, row in documents_df.iterrows():
    for chunk_index, chunk in enumerate(chunk_text(row["text"])):
        chunks.append(
            {
                "id": f"doc-{row_index}-chunk-{chunk_index}",
                "text": chunk,
                "title": row["title"],
                "url": row["url"],
            }
        )

print(f"Created {len(chunks)} chunks.")

## Embed and index the chunks

Gemini creates an embedding for each passage. ChromaDB stores the vectors and metadata, then retrieves passages most similar to a query.

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.8-flash"


def embed_texts(texts: list[str]) -> list[list[float]]:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
    )
    return [embedding.values for embedding in response.embeddings]


chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="medlineplus_passages",
    metadata={"hnsw:space": "cosine"},
)

chunk_texts = [item["text"] for item in chunks]
chunk_embeddings = embed_texts(chunk_texts)

collection.upsert(
    ids=[item["id"] for item in chunks],
    documents=chunk_texts,
    embeddings=chunk_embeddings,
    metadatas=[
        {"title": item["title"], "url": item["url"]}
        for item in chunks
    ],
)

print(f"Indexed {collection.count()} chunks.")

## Try retrieval

In [ ]:
def retrieve(query: str, top_k: int = 4) -> list[dict]:
    query_embedding = embed_texts([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for text, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append(
            {
                "text": text,
                "title": metadata["title"],
                "url": metadata["url"],
                "distance": distance,
            }
        )
    return retrieved


query = "What does reduced ejection fraction mean?"
retrieved_passages = retrieve(query)

for index, passage in enumerate(retrieved_passages, start=1):
    print(f"{index}. {passage['title']} (distance: {passage['distance']:.3f})")
    print(passage["text"])
    print(passage["url"])
    print()

## Generate a simplification

In [ ]:
def simplify_with_retrieval(clinical_source: str, top_k: int = 4) -> tuple[str, list[dict]]:
    passages = retrieve(clinical_source, top_k=top_k)
    evidence = "\n\n".join(
        f"Passage {index}: {item['text']}"
        for index, item in enumerate(passages, start=1)
    )

    prompt = f"""
Rewrite the clinical source in clear, patient-friendly language.

Use the clinical source as the only authority for facts about this patient.
Use the reference passages only to explain medical terms in plain language.
Do not add a diagnosis, treatment, or patient-specific fact that is not in the clinical source.
Preserve uncertainty, numbers, negation, and follow-up instructions.
If a term is not explained by the references, do not guess.

Clinical source:
{clinical_source}

Retrieved reference passages:
{evidence}
"""

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
    )
    return response.text or "", passages


clinical_source = """
The echocardiogram showed a reduced ejection fraction of 35%.
"""

answer, cited_passages = simplify_with_retrieval(clinical_source)

print("Simplified explanation:")
print(answer)

print("\nRetrieved references:")
for passage in cited_passages:
    print(f"- {passage['title']}: {passage['url']}")

## Record retrieval quality

Test several queries from the team's examples. For each one, note whether the retrieved passages are relevant, understandable, and from a suitable source.